In [10]:
import xarray as xr
import numpy as np
import os
import datetime
from datetime import date
import era5_config as cfg
import helper_functions as helper
import cartopy.crs as ccrs 
import matplotlib.pyplot as plt

In [23]:
data = xr.open_dataset(os.path.join('/cpc/int_desk/era5/data/raw/nc_files', 'dailysingle_africa_t2m_2025-02-07_ERA5.nc'))
data = data.rename({'latitude':'y', 'longitude':'x'})
data = data.sortby('y', ascending = True)
data

<xarray.Dataset>
Dimensions:     (valid_time: 73656, y: 321, x: 301)
Coordinates:
    number      int64 ...
  * valid_time  (valid_time) datetime64[ns] 1991-01-01 ... 2024-12-31T23:00:00
  * y           (y) float64 -40.0 -39.75 -39.5 -39.25 ... 39.25 39.5 39.75 40.0
  * x           (x) float64 -20.0 -19.75 -19.5 -19.25 ... 54.25 54.5 54.75 55.0
    expver      (valid_time) object ...
Data variables:
    t2m         (valid_time, y, x) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-02-07T17:46 GRIB to CDM+CF via cfgrib-0.9.1...

In [22]:
data.isel(valid_time=0).t2m.sel(x=slice(-20,-10), y = slice(0,20)).values

array([[299.19244, 299.1358 , 299.06744, ..., 298.27643, 298.33112,
        298.3897 ],
       [299.50885, 299.38385, 299.23932, ..., 298.25104, 298.30377,
        298.33307],
       [299.7354 , 299.71393, 299.5733 , ..., 298.294  , 298.35065,
        298.37408],
       ...,
       [293.5987 , 293.48737, 293.39948, ..., 290.4112 , 289.9698 ,
        289.5069 ],
       [293.55573, 293.4151 , 293.3194 , ..., 290.4776 , 289.92682,
        289.46588],
       [293.52643, 293.38385, 293.26276, ..., 290.29987, 289.8194 ,
        289.49127]], dtype=float32)

In [31]:
today = date.today()
download_file_name = '_'.join(['dailysingle',cfg.var, str(today), str('ERA5.nc')])
data = xr.open_dataset(os.path.join(cfg.download_raw_data_dir, 'nc_files', download_file_name))
data = data.rename({'longitude': 'lon', 'latitude': 'lat', 'time': 'T'})

In [34]:
most_recent_date = np.datetime64(data.T.max().values)
prior_week = datetime.datetime(most_recent_date) - datetime.timedelta(days = 7)
prior_week

/tmp/ipykernel_14504/1481349885.py:2: DeprecationWarning: an integer is required (got type numpy.datetime64).  Implicit conversion to integers using __int__ is deprecated, and may be removed in a future version of Python.
  prior_week = datetime.datetime(most_recent_date) - datetime.timedelta(days = 7)


OverflowError: signed integer is greater than maximum

In [43]:
data.T.max()

<xarray.DataArray 'T' ()>
array('2024-05-23T00:00:00.000000000', dtype='datetime64[ns]')

In [ ]:
data['anom'] = data['t2m'].groupby('T.month').apply(helper.calc_anomaly)
data